In [1]:
# Import the required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
# Define transformations for the train and validation sets
transform = transforms.Compose([
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),  # Resize the images to match ResNet input
    transforms.ToTensor(),          # Convert images to PyTorch tensors
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize with ImageNet stats
])

# Download and load the training and validation datasets
train_dataset = datasets.Flowers102(root='data', split='train', download=True, transform=transform)
val_dataset = datasets.Flowers102(root='data', split='val', download=True, transform=transform)

# Create DataLoader for batch processing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [3]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 224 → 112

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 112 → 56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),     # 56 → 28

            # Global pooling
            nn.AdaptiveAvgPool2d((1,1)),

            nn.Flatten()
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [7]:
model_scratch = FlowerCNN().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_scratch.parameters(),
    lr=0.001
)


In [5]:
# Training function
def train_model_scratch(model_scratch, train_loader, loss_fn, optimizer, num_epochs=8):
    model_scratch.train()

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_scratch(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

# Train the model from scratch
train_model_scratch(model_scratch, train_loader, loss_fn, optimizer, num_epochs=8)


Epoch [1/8], Loss: 4.6277, Accuracy: 1.76%
Epoch [2/8], Loss: 4.4849, Accuracy: 4.31%
Epoch [3/8], Loss: 4.2980, Accuracy: 6.27%
Epoch [4/8], Loss: 4.1008, Accuracy: 5.69%
Epoch [5/8], Loss: 3.9188, Accuracy: 8.33%
Epoch [6/8], Loss: 3.7724, Accuracy: 8.63%
Epoch [7/8], Loss: 3.6442, Accuracy: 12.45%
Epoch [8/8], Loss: 3.5350, Accuracy: 12.75%


In [6]:
def evaluate_model(model_scratch, val_loader):
    # Evaluation loop
    model_scratch.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_scratch(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

evaluate_model(model_scratch, val_loader)

Test Accuracy: 17.16%


In [8]:
# Load pre-trained ResNet18 model with proper weights specification
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all the convolutional layers (feature extractor)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer (for 1000 ImageNet classes) with one for 102 flowers
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 102)

# Move the model to the GPU if available
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Asus/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [9]:
# Define loss function and optimizer (only for the last layer)
loss_fn = nn.CrossEntropyLoss()

# Since we're only training the final layer, only pass its parameters to the optimizer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [10]:
# Training function
def train_model(model, train_loader, loss_fn, optimizer, num_epochs=6):
    model.train()  # Set the model to training mode

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        # Iterate over batches
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Track training loss
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # Print statistics after each epoch
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

# Train the model for 6 epochs
train_model(model, train_loader, loss_fn, optimizer, num_epochs=6)


Epoch [1/6], Loss: 4.6606, Accuracy: 2.25%
Epoch [2/6], Loss: 3.4088, Accuracy: 35.29%
Epoch [3/6], Loss: 2.5767, Accuracy: 61.47%
Epoch [4/6], Loss: 1.9063, Accuracy: 78.82%
Epoch [5/6], Loss: 1.4788, Accuracy: 85.69%
Epoch [6/6], Loss: 1.1910, Accuracy: 88.24%


In [11]:
# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients for validation
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Validation Accuracy: {accuracy:.2f}%')

# Evaluate the model
evaluate_model(model, val_loader)

Validation Accuracy: 76.67%
